# VLA 연구 · 12회차 — VLM → VLA 실행 파이프라인

| 항목 | 내용 |
|---|---|
| 오늘 목표 | `이미지 + 명령` → **VLM이 순차 지시로 분해** → **VLA가 관절 제어로 실행** 을 끝까지 돌린다 |
| 환경 | MuJoCo + Franka Panda, 관측·액션은 **LIBERO 호환** 규격 |
| 산출물 | 파이프라인 rollout GIF, VLM 계획 채점표, 텔레옵 데이터셋, SmolVLA 학습 준비 |

지금까지와 달라진 점: 10·11회차의 `MiniVLAController` 는 물체 좌표를 목표까지 **보간**해서
옮기는 흉내였습니다. 이번에는 **실제로 그리퍼로 집습니다** (물리 접촉·마찰 기반).


---
## SECTION 0 | 설치와 코드 연결

`vla_pipeline` 폴더가 팀 공유 드라이브에 있어야 합니다.

```
MyDrive/VLA_Team_2026/
└── vla_pipeline/          <- make_student_zip.py 로 만든 zip 을 푼 것
    ├── vlapipe/  scripts/  assets/  notebooks/
    └── README.md ...
```

**공유 폴더에서 바로 돌리지 않습니다.** 아래 0-2 가 코드를 런타임 로컬(`/content`)로
복사한 뒤 거기서 작업합니다. 이유는 셀 주석에 적어 두었습니다.
결과물은 마지막 셀에서 다시 공유 드라이브로 올립니다.


In [1]:
# 0-1. 패키지 설치 (Colab 최초 1회, 이후 런타임 재시작)
!pip install -q 'mujoco>=3.2,<4.0' h5py imageio imageio-ffmpeg 'transformers>=4.50' accelerate num2words torchvision
print('설치 완료 — Runtime > Restart session 후 0-2 부터 실행하세요.')

In [ ]:
# 0-2. Drive 마운트 + 코드를 런타임 로컬로 복사
import os, sys, shutil
os.environ['MUJOCO_GL'] = 'egl'
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    TEAM = Path('/content/drive/MyDrive/VLA_Team_2026')
    SRC = TEAM / 'vla_pipeline'
    assert SRC.exists(), f'공유 드라이브에 없습니다: {SRC}'

    # 공유 폴더에서 바로 돌리면 안 되는 이유:
    #  1) SortingEnv 는 만들 때마다 assets/.../vla_scene.xml 을 새로 씁니다.
    #     셋이 동시에 돌리면 서로의 씬 파일을 덮어써 ParseXML 오류가 납니다.
    #  2) 학습 체크포인트(outputs/smolvla_sorting)가 GB 단위로 쌓여
    #     공유 드라이브 용량을 먹습니다.
    #  3) Drive(FUSE)는 느려서 메시 파일 로딩이 오래 걸립니다.
    # -> 코드와 에셋만 런타임 로컬로 복사해 거기서 작업합니다. 원본은 건드리지 않습니다.
    PKG = Path('/content/vla_pipeline')
    shutil.rmtree(PKG / 'data', ignore_errors=True)      # 새로 올린 녹화본을 반영
    shutil.copytree(SRC, PKG, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.venv', 'outputs', '__pycache__', '.git'))
    print('코드 사본 :', PKG)
    print('원본      :', SRC, '(읽기만 함)')
else:
    PKG = Path.cwd().parent
    TEAM = PKG.parent

assert PKG.exists(), f'경로를 찾을 수 없습니다: {PKG}'
sys.path.insert(0, str(PKG))
os.chdir(PKG)
(PKG / 'outputs').mkdir(exist_ok=True)

n_ep = len(list((PKG / 'data' / 'episodes').glob('*.hdf5'))) if (PKG / 'data' / 'episodes').exists() else 0
print('작업 폴더 :', PKG)
print('녹화본    :', n_ep, '개')


In [3]:
# 0-3. Panda 에셋 준비 (최초 1회, 약 40MB)
!python setup_assets.py

In [4]:
# 0-4. 환경 점검
!python scripts/check_env.py

---
## SECTION 1 | 씬 살펴보기

260827 문서에서 확정한 규격 그대로입니다: 작업 영역 0.8 × 0.5 m, 3×1 구역, 물체 10개, 카메라 3대 + 손목.

**정돈된 상태** = 긴 블록 5개는 오른쪽 보관 구역, 정육면체·타일 5개는 왼쪽 보관 구역, 가운데는 비어 있음.


In [5]:
import numpy as np, matplotlib.pyplot as plt
from vlapipe.env import SortingEnv
from vlapipe import config as C

env = SortingEnv(layout='tidy', image_size=256, cameras=C.ALL_CAMERAS)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for r, (lay, seed) in enumerate([('tidy', 0), ('messy', 3)]):
    env.reset(layout=lay, seed=seed)
    for c, cam in enumerate(C.ALL_CAMERAS):
        axes[r, c].imshow(env.render(cam, 300, 300)); axes[r, c].axis('off')
        axes[r, c].set_title(f'{lay} / {cam}', fontsize=9)
plt.tight_layout(); plt.show()
print('잘못 놓인 물체:', env.misplaced_objects())

### 왜 `agentview` 는 위에서 내려다보나요?

문서의 `frontview`(로봇을 마주보는 면)는 **영상의 좌우가 월드 좌우와 뒤집힙니다.**
그 시점으로 학습하면 VLM 이 말하는 "왼쪽"과 로봇 기준 "왼쪽"이 반대가 됩니다.
그래서 학습·추론용 주 카메라는 top-down 으로 두었습니다 (영상 오른쪽 = 월드 +x).

> **생각해 볼 것**: 실제 로봇에 옮길 때 천장 카메라를 달 수 없다면 어떻게 해야 할까요?


---
## SECTION 2 | 액션과 관측 규격 (LIBERO 호환)

```
action = [dx, dy, dz, drx, dry, drz, gripper]      # 7차원
  앞 3개 : 위치 델타, [-1,1] -> 최대 ±0.05 m
  중간 3개: 회전 델타(축각), [-1,1] -> 최대 ±0.5 rad
  마지막 : 그리퍼, -1 = 열기 / +1 = 닫기
```
이 규격을 LIBERO 와 맞춰 두면 나중에 LIBERO 사전학습 체크포인트나 데이터셋을 그대로 붙일 수 있습니다.


In [6]:
obs = env.reset(layout='messy', seed=3)
for k, v in obs.items():
    print(f'  {k:28s} {np.asarray(v).shape} {np.asarray(v).dtype}')
print('\nstate 벡터:', SortingEnv.state_vector(obs).shape)
print('제어 주기:', C.CONTROL_HZ, 'Hz | 액션당 시뮬 스텝:', C.SIM_STEPS_PER_ACTION)

In [7]:
# 액션을 직접 넣어 보기 — 그리퍼를 앞(+y)으로 밀고 내린다
env.reset(layout='messy', seed=3)
print('시작 eef:', env.ctrl.eef_pos().round(4))
for _ in range(6):
    env.step([0.0, 0.6, -0.5, 0, 0, 0, -1.0])
print('이후 eef:', env.ctrl.eef_pos().round(4))
plt.figure(figsize=(4,4)); plt.imshow(env.render('agentview', 300, 300)); plt.axis('off'); plt.show()

---
## SECTION 3 | 1단계 — VLM 플래너

`RulePlanner` 는 시뮬레이터 상태를 직접 보고 만든 **정답 계획**입니다. VLM 채점 기준으로 씁니다.


In [8]:
from vlapipe.planner import RulePlanner, VLMPlanner, plan_accuracy, parse_plan

env.reset(layout='messy', seed=3)
gold = RulePlanner(env).plan('테이블을 정리하시오')
for s in gold:
    print(f"  {s['step']}. {s['instruction']}")
    print(f"     {s['instruction_ko']}   -> ({s['object']}, {s['target_zone']})")

In [9]:
# SmolVLM2 로 같은 장면을 계획하게 한다 (최초 실행 시 모델 다운로드)
img = env.render('agentview', 448, 448)
vlm = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct')  # 기본 prompt_style='json'
raw = vlm.raw_response(img, '테이블을 정리하시오')
print('=== VLM 원문 ==='); print(raw)
pred = parse_plan(raw)
print('\n=== 파싱된 계획 ==='); [print('  ', s['step'], s['object'], '->', s['target_zone']) for s in pred]
print('\n=== 채점 ==='); print(plan_accuracy(pred, gold))

### ✏️ 미니과제 — 프롬프트 배틀 (7회차의 확장판)

조교가 미리 재 본 값입니다 (씬 3개, RTX 4080):

| 모델 | 프롬프트 | recall | 추론 |
|---|---|---|---|
| SmolVLM2-500M | `json` | 0.23 | 1.5 s |
| SmolVLM2-500M | `lines` | 0.00 | 0.5 s |
| SmolVLM2-2.2B | `json` | ~0.31 | **55 s** |

여기서 이미 세 가지가 보입니다.

1. **2.2B 는 값어치를 못 합니다** — 정확도는 조금 오르는데 추론이 40배 느립니다.
2. **2.2B 는 이미지를 거의 안 봅니다** — 씬을 바꿔도 출력 앞부분이 거의 똑같습니다.
   프롬프트에 쓴 규칙을 베껴 쓰는 것에 가깝습니다.
3. **`lines` 는 실패했습니다** — "한 줄에 하나씩"이 작은 모델에 쉬울 것 같지만,
   실제로는 한 줄만 쓰고 멈춥니다. 추측이 실측으로 뒤집힌 사례입니다.

**여러분이 할 일**: `vlapipe/planner.py` 의 `_PROMPT` 를 고쳐 이 표를 갱신하세요.
팀원 3명이 서로 다른 전략으로 붙어 봅니다.

- 구역을 색으로 부를까(초록/파랑), 위치로 부를까(왼쪽/오른쪽)?
- 물체 목록을 프롬프트에 미리 알려 줄까?
- 이미지 해상도(256 / 384 / 448)를 바꾸면 달라지나?
- "가운데에 무엇이 있나?" 만 먼저 묻고 목적지는 규칙으로 정하면?

> **이 결과가 보고서에 들어갑니다.** 3번 관찰(모델이 이미지를 안 본다)은
> "왜 VLM 을 파인튜닝해야 하는가"의 직접적인 근거입니다.


In [ ]:
# 여러 씬에서 VLM 계획 정확도를 재는 채점 루프
import pandas as pd
rows = []
for seed in [3, 7, 11, 19, 23]:
    env.reset(layout='messy', seed=seed)
    g = RulePlanner(env).plan()
    p = parse_plan(vlm.raw_response(env.render('agentview', 448, 448), '테이블을 정리하시오'))
    a = plan_accuracy(p, g)
    rows.append({'seed': seed, '정답수': a['n_gold'], '예측수': a['n_pred'],
                 '맞춘수': a['correct'], 'recall': round(a['recall'], 2),
                 'precision': round(a['precision'], 2)})
df = pd.DataFrame(rows); display(df)
print('평균 recall %.2f  precision %.2f' % (df.recall.mean(), df.precision.mean()))

In [ ]:
# 프롬프트 A/B 비교 — 같은 씬에 두 프롬프트를 붙여 본다
from vlapipe.planner import PROMPTS

def score_prompt(style, seeds=(3, 7, 11), img_size=448):
    vp = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style=style)
    c = g = n = 0
    for sd in seeds:
        env.reset(layout='messy', seed=sd)
        gold = RulePlanner(env).plan()
        pred = parse_plan(vp.raw_response(env.render('agentview', img_size, img_size), '테이블을 정리하시오'))
        a = plan_accuracy(pred, gold)
        c += a['correct']; g += a['n_gold']; n += a['n_pred']
    del vp
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
    return {'프롬프트': style, 'recall': round(c / g, 2),
            'precision': round(c / n, 2) if n else 0.0, '맞춘수': c, '정답수': g, '예측수': n}

display(pd.DataFrame([score_prompt(s) for s in PROMPTS]))

---
## SECTION 3.5 | 숫자로 물체를 구분하기 (`--objects number`)

여기까지는 물체를 **색과 모양**으로 불렀습니다 ("빨간 블록", "긴 블록").
그러면 VLM 은 숫자를 읽지 않고도 색만 보고 찍을 수 있습니다.

이 소절은 **구분 단서만 바꾼** 조건입니다.

| | 색 조건 (`color`) | 숫자 조건 (`number`) |
|---|---|---|
| 물체 10개 | 색·크기·모양이 서로 다름 | **전부 똑같은 흰 정육면체** (0.05 m) |
| 구분 단서 | 색 + 모양 | 면에 찍힌 숫자 **1~10** 뿐 |
| 정돈 규칙 | 긴 블록 → 오른쪽 / 정육면체·타일 → 왼쪽 | **1~5 → 오른쪽 / 6~10 → 왼쪽** |
| 지시문 | "Pick up the red block …" | "Pick up the cube number 7 …" |

**바뀌지 않는 것**: 테이블·구역·조명·카메라·로봇, 배치 생성(layout)과 시드,
그리고 LIBERO 호환 규격(7차원 델타 액션 · 16차원 state). 이게 같아야 비교가 성립합니다.

### 왜 이 조건을 만드는가

색은 **한 픽셀만 봐도** 알 수 있지만, 숫자는 **글자 모양을 읽어야** 알 수 있습니다.
숫자 조건에서 성능이 떨어진다면 그건 "물체를 옮기는 게 어려워서"가 아니라
**"무엇을 옮길지 알아보지 못해서"** 입니다. 즉 1단계(VLM)의 한계를 2단계(VLA)와
섞이지 않게 떼어내서 재는 장치입니다.

> 이 소절은 SECTION 1~3 셀을 먼저 돌린 상태(`env`, `pd`, `plt`, `RulePlanner` …)를 전제합니다.
> **GPU 런타임(T4)** 이 필요합니다 — `런타임 > 런타임 유형 변경 > T4 GPU`.

In [ ]:
# 3.5-1. 숫자 조건으로 바꾸고 env 를 "새로" 만든다 + 256px / 512px 비교
from vlapipe import numbered
from vlapipe.env import SortingEnv
from vlapipe import config as C

# activate() 는 숫자 텍스처 10장을 만들어 씬에 등록하고 물체 세트를 갈아끼운다.
# 반드시 SortingEnv 보다 "먼저" 불러야 한다 — env 는 만들어질 때 물체 이름으로
# MuJoCo 조인트 주소를 잡아 두기 때문에, 나중에 세트를 바꾸면 이름이 맞지 않는다.
numbered.activate()
env_num = SortingEnv(layout='messy', seed=3, image_size=C.IMAGE_SIZE, cameras=C.ALL_CAMERAS)
env_num.reset(layout='messy', seed=3)

# 숫자 세트가 켜진 동안에는 앞에서 만든 color 용 env 를 reset/render 하지 말 것.
# (색 조건으로 돌아가려면 3.5-4 처럼 numbered.deactivate() 를 먼저 부른다.)
print('물체 세트:', C.ACTIVE_OBJECT_SET, '| 물체 10개:', C.OBJECT_NAMES)
print('정돈 규칙: 1~%d -> 오른쪽, %d~%d -> 왼쪽'
      % (C.NUMBER_SPLIT, C.NUMBER_SPLIT + 1, C.NUMBER_COUNT))
print('잘못 놓인 물체:', env_num.misplaced_objects())

# 그림 제목은 영어로 둔다 — Colab 의 matplotlib 에는 한글 글꼴이 없어서
# 한글을 넣으면 네모(□□□)로 나온다. 설명은 아래 print 로 뺀다.
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, px in zip(axes, (256, 512)):
    ax.imshow(env_num.render('agentview', px, px), interpolation='nearest')
    ax.axis('off')
    # face_pixels() 는 카메라 기하로 "계산한" 값이다 (렌더 실측이 아니다).
    # 숫자 자체의 높이는 윗면 한 변의 약 0.7 배.
    ax.set_title(f'agentview {px}px  (top face ~{numbered.face_pixels(px):.1f} px)',
                 fontsize=11)
plt.suptitle('same scene, render resolution only')
plt.tight_layout(); plt.show()

print('\n위: 같은 장면을 256px / 512px 로 렌더한 것입니다.')
print('화면에서는 둘을 같은 크기로 늘려 그렸으므로, 흐린 쪽이 곧 정보가 없는 쪽입니다.')
print('어느 쪽에서 숫자가 읽히나요? 256px 에서 읽히지 않는다면, VLM 에게 256px 을 주는 것은')
print('"숫자를 지우고 물체를 구분해 보라"고 하는 것과 같습니다.')
print('(조교가 본 바로는 512px 에서 또렷하고 256px 에서는 읽기 어려웠습니다 — 사람 눈 기준이고,')
print(' VLM 이 어디서부터 못 읽는지는 아직 아무도 재지 않았습니다. 3.5-3 에서 직접 재세요.)')
print('가운데 구역만 확대한 그림이 필요하면: !python scripts/check_numbers.py --rollout 0')

In [ ]:
# 3.5-2. 한 장면 — 정답 계획(규칙 플래너) vs VLM 계획(프롬프트 'number')
from vlapipe.planner import RulePlanner, VLMPlanner, plan_accuracy, parse_plan, PROMPTS

# numbered 를 임포트하면 숫자용 프롬프트 두 개가 planner.PROMPTS 에 등록된다.
print('쓸 수 있는 프롬프트:', sorted(PROMPTS))

# 주의: 이제 PROMPTS 에 'number'/'number_read' 가 들어 있으므로, 위쪽 '프롬프트 배틀' 셀의
# `for s in PROMPTS` 루프를 다시 돌리면 색 조건에 숫자 프롬프트까지 붙여 버린다.
# 그 셀을 다시 돌릴 때는 `for s in ['json', 'lines']` 로 바꾸거나 런타임을 새로 시작하세요.

# 숫자 조건의 VLM 입력 해상도. 3.5-1 에서 본 이유로 512px 을 쓴다.
# 관측(obs)·학습 데이터 해상도인 C.IMAGE_SIZE(=256, LIBERO 기본값)는 건드리지 않는다.
VLM_PX = 512

env_num.reset(layout='messy', seed=3)
gold_num = RulePlanner(env_num).plan('테이블을 정리하시오')

vlm_num = VLMPlanner('HuggingFaceTB/SmolVLM2-500M-Video-Instruct', prompt_style='number')
raw_num = vlm_num.raw_response(env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오')
print('\n=== VLM 원문 ===');  print(raw_num)

pred_num = parse_plan(raw_num)
acc_num = plan_accuracy(pred_num, gold_num)


def _fmt(s):
    return '' if s is None else f"{s['object']} -> {s['target_zone']}"


print('\n{:<28} | {}'.format('정답 (규칙 플래너)', 'VLM 예측'))
print('-' * 58)
for i in range(max(len(gold_num), len(pred_num))):
    g = gold_num[i] if i < len(gold_num) else None
    p = pred_num[i] if i < len(pred_num) else None
    print('{:<28} | {}'.format(_fmt(g), _fmt(p)))

print('\n=== 채점 ===')
for k, v in acc_num.items():
    print(f'  {k:14s} {v}')

In [ ]:
# 3.5-3. 채점 루프 — 시드 여러 개 × 프롬프트 두 가지
# 16번 셀(색 조건 채점 루프)과 같은 틀이다. 다른 점은 세 가지뿐:
#   (a) 숫자 조건 env_num,  (b) 512px 입력,  (c) 프롬프트 'number' / 'number_read'
#
#   'number'      : 숫자를 읽고 + 규칙(1~5 오른쪽 / 6~10 왼쪽)까지 적용하게 한다
#   'number_read' : 가운데에 있는 숫자만 읽게 하고, 목적지는 코드가 규칙으로 정한다
# -> 두 점수의 차이가 "숫자를 못 읽는 것"과 "규칙을 못 쓰는 것"을 갈라 준다.
#    (messy 배치에서는 잘못 놓인 물체가 전부 가운데에 있으므로 두 프롬프트의
#     정답(gold)이 같다. 그래서 점수를 그대로 맞대어 볼 수 있다.)

SEEDS_NUM = [3, 7, 11, 19, 23]        # 수업 중에는 5개 (T4 에서 몇 분)
# SEEDS_NUM = list(range(30))         # 보고서용: 조건당 30씬. 색 조건과 '같은 시드'를 써야
                                      # 비교가 됩니다. 시간이 오래 걸리니 수업 뒤에 돌리세요.

rows_num = []
for style in ['number', 'number_read']:
    vlm_num.prompt_style = style      # 같은 가중치, 프롬프트만 교체 (모델을 다시 올리지 않는다)
    for seed in SEEDS_NUM:
        env_num.reset(layout='messy', seed=seed)
        g = RulePlanner(env_num).plan()
        p = parse_plan(vlm_num.raw_response(
            env_num.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'))
        a = plan_accuracy(p, g)
        rows_num.append({'프롬프트': style, 'seed': seed,
                         '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                         'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                         '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                         '구역틀림': len(a['wrong_zone'])})

df_num = pd.DataFrame(rows_num)
display(df_num)


def micro(df, label):
    """씬별 recall 을 그냥 평균내면 정답이 1개뿐인 씬이 과하게 센다.
    그래서 맞춘수/정답수를 전체 합으로 다시 계산한다 (micro 평균)."""
    c, g, n = int(df['맞춘수'].sum()), int(df['정답수'].sum()), int(df['예측수'].sum())
    return {'조건': label, '씬수': len(df), '정답수': g, '예측수': n, '맞춘수': c,
            'recall': round(c / g, 2) if g else float('nan'),
            'precision': round(c / n, 2) if n else float('nan'),
            '빠뜨림': int(df['빠뜨림'].sum()), '없는물체': int(df['없는물체'].sum()),
            '구역틀림': int(df['구역틀림'].sum())}


display(pd.DataFrame([micro(df_num[df_num['프롬프트'] == s], f'숫자 ({s})')
                      for s in ['number', 'number_read']]))
print('빠뜨림 = 정답에 있는데 말하지 않음')
print('없는물체 = 옮길 필요 없는 것을 옮기라고 함')
print('구역틀림 = 물체는 맞고 목적지가 틀림')

In [ ]:
# 3.5-4. 색 조건과 '같은 시드'로 비교
# deactivate() 로 색 세트로 되돌린다. 앞에서 만든 env 는 색 세트가 켜진 상태에서
# 만들어졌으므로, 되돌린 뒤에는 그대로 다시 쓸 수 있다.
numbered.deactivate()
print('물체 세트:', C.ACTIVE_OBJECT_SET)

# 공정한 비교를 위해 색 조건도 같은 512px 로 다시 잰다.
# (16번 셀은 448px 이었으니 숫자가 조금 다르게 나올 수 있습니다. 해상도가 다르면
#  '구분 단서' 말고 다른 것이 섞여 버리므로, 여기서 다시 재는 쪽이 맞습니다.)
vlm_num.prompt_style = 'json'         # 색 조건의 기본 프롬프트

rows_col = []
for seed in SEEDS_NUM:
    env.reset(layout='messy', seed=seed)
    g = RulePlanner(env).plan()
    p = parse_plan(vlm_num.raw_response(
        env.render('agentview', VLM_PX, VLM_PX), '테이블을 정리하시오'))
    a = plan_accuracy(p, g)
    rows_col.append({'프롬프트': 'json', 'seed': seed,
                     '정답수': a['n_gold'], '예측수': a['n_pred'], '맞춘수': a['correct'],
                     'recall': round(a['recall'], 2), 'precision': round(a['precision'], 2),
                     '빠뜨림': len(a['missed']), '없는물체': len(a['hallucinated']),
                     '구역틀림': len(a['wrong_zone'])})

df_col = pd.DataFrame(rows_col)
display(df_col)

display(pd.DataFrame([
    micro(df_col, '색 (json)'),
    micro(df_num[df_num['프롬프트'] == 'number'], '숫자 (number)'),
    micro(df_num[df_num['프롬프트'] == 'number_read'], '숫자 (number_read)'),
]))

print('같은 시드 · 같은 layout(messy) · 같은 모델 · 같은 512px 입력입니다.')
print('다른 것은 구분 단서(색/숫자)와 프롬프트뿐입니다.')
print('\n※ 씬 5개로는 결론을 내릴 수 없습니다 — recall 0.2 와 0.4 의 95% 신뢰구간이 겹칩니다.')
print('   보고서에는 SEEDS_NUM 을 30개로 늘려서 다시 재세요.')

# 숫자 조건으로 돌아가려면 numbered.activate() 를 부르고 env 를 "새로" 만들 것.
# (SECTION 4 부터는 색 조건으로 이어집니다.)

### ✏️ 미니과제 — 숫자를 못 읽는 것 vs 규칙을 못 쓰는 것

VLM 이 틀렸을 때 "그냥 못한다"로 끝내면 다음에 무엇을 고쳐야 할지 알 수 없습니다.
두 프롬프트를 나란히 재는 이유가 여기 있습니다.

| 프롬프트 | VLM 이 하는 일 | 규칙(1~5 / 6~10)은 누가 적용? |
|---|---|---|
| `number` | 숫자를 읽고 **목적지까지** 말한다 | VLM |
| `number_read` | 가운데 있는 숫자만 읽는다 | **코드** |

읽는 법:

- `number_read` 도 낮다 → **숫자를 못 읽는 것**입니다. 1단계의 *지각* 문제입니다.
  해상도를 올리거나, 숫자를 더 크고 또렷하게 하거나, VLM 을 파인튜닝해야 합니다.
- `number_read` 는 높은데 `number` 가 낮다 → **읽기는 되는데 규칙을 못 쓰는 것**입니다.
  1단계의 *추론* 문제이고, 프롬프트를 고치거나 목적지를 코드에 맡겨 해결할 수 있습니다.
- 둘 다 색 조건(`json`)보다 많이 낮다 → 색 조건의 점수에는 **단서가 쉬워서 얻은 점수**가
  섞여 있었다는 뜻입니다. 이게 이 소절이 재려는 바로 그 값입니다.

#### 여러분이 채울 표

아직 **아무도 숫자 조건의 VLM 정확도를 재지 않았습니다.** 위 셀을 직접 돌려서 채우세요.
(채우기 전에는 빈칸으로 두세요. 추측한 값을 실측값처럼 적으면 보고서가 무너집니다.)

| 조건 | 프롬프트 | 입력 px | 씬 수 | recall | precision | 빠뜨림 | 없는물체 | 구역틀림 |
|---|---|---|---|---|---|---|---|---|
| 색 | `json` | 512 | | | | | | |
| 숫자 | `number` | 512 | | | | | | |
| 숫자 | `number_read` | 512 | | | | | | |
| 숫자 | `number` | 256 | | | | | | |

- 마지막 줄(256px)은 `VLM_PX = 256` 으로 바꿔 3.5-3 을 한 번 더 돌린 것입니다.
  **해상도를 반으로 줄이면 점수가 얼마나 떨어지는가** — 숫자를 읽는 것이 이 과제의
  병목인지 확인하는 가장 빠른 방법입니다.
- 표 아래에 **2~3문장**으로 적으세요: 어느 쪽 문제였나? 무엇을 근거로 그렇게 보았나?

#### 더 해 볼 것

1. 규칙 플래너로 숫자 조건을 돌려, 2단계(VLA)는 색 조건과 비슷하게 되는지 확인하세요.
   비슷해야 "1단계만의 문제"라고 말할 수 있습니다.

   ```
   !python scripts/run_pipeline.py --objects number --layout messy --seed 3 --planner rule
   ```

2. 틀린 숫자를 적어 보세요. **6 과 9**, **1 과 7** 처럼 헷갈리는 짝이 많으면
   글자 모양 문제이고, 아무 숫자나 찍는다면 아예 읽지 않고 있는 것입니다.
   (`vlapipe/numbered.py` 는 6/9 구분을 돕는 밑줄을 숫자 아래에 그려 둡니다.)

3. 조건을 하나 더 만들고 싶으면 `config.OBJECT_SETS` 에 세트를 추가하면 됩니다
   (`ref_en`/`ref_ko`/`patterns` 만 채우면 플래너·정책은 그대로 돌아갑니다).

---
## SECTION 4 | 2단계 — VLA 정책

`ScriptedSortingPolicy` 는 **teacher(선생) 정책**입니다. 물체 위치를 시뮬레이터에서 직접 읽으므로
학습된 정책이 아닙니다. 두 가지 용도로 씁니다:

1. SmolVLA 를 학습시키기 전에도 파이프라인이 끝까지 돌게 하는 **대역**
2. 학습 데이터를 자동 생성하는 **데모 생성기**

인터페이스는 SmolVLA 와 똑같은 `select_action()` 이라서 나중에 그대로 갈아끼웁니다.


In [ ]:
from vlapipe.policy import ScriptedSortingPolicy
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display

env.reset(layout='messy', seed=3)
policy = ScriptedSortingPolicy(env)
frames, log = [], []
for st in RulePlanner(env).plan():
    policy.set_subtask(st); n = 0
    while not policy.done and n < 500:
        env.step(policy.select_action()); n += 1
        if n % 3 == 0: frames.append(env.render('frontview', 300, 400))
    ok = env.zone_of(st['object']) == st['target_zone']
    log.append({'물체': st['object'], '스텝': n, '성공': ok, '실패사유': policy.failure})
display(pd.DataFrame(log))
print('정리 완료:', env.is_tidy(), '| 남은:', env.misplaced_objects())
imageio.mimsave('outputs/section4.gif', frames[::2], fps=14, loop=0)
display(IPyImage(filename='outputs/section4.gif'))

---
## SECTION 5 | 엔드투엔드 실행

한 번에 돌리는 스크립트입니다. 실패한 물체는 다음 패스에서 다시 시도합니다.


In [ ]:
!python scripts/run_pipeline.py --layout messy --seed 11 --passes 2 --planner rule

In [ ]:
# VLM 플래너로도 돌려 보기 (계획은 VLM, 실행은 scripted)
!python scripts/run_pipeline.py --layout messy --seed 11 --passes 2 --planner vlm

---
## SECTION 6 | 데이터 — 어지르고 나서 뒤집기

정확한 위치에 **내려놓기**는 손으로 어렵지만, 구역에서 **집어서 아무 데나 놓기**는 쉽습니다.
그래서 정돈된 씬을 어지르는 쪽을 게임패드로 녹화하고, 시간 역순으로 뒤집어 정돈 데모로 씁니다.

**녹화는 윈도우 로컬에서** (`scripts/teleop_record.py`), 뒤집기와 학습은 여기 Colab 에서 합니다.

### 뒤집기의 수학
`target[t+1] = target[t] + a[t]` 이므로 위치·회전 델타는 부호만 뒤집으면 됩니다:
`a'[k][:6] = -a[T-1-k][:6]`

그리퍼는 델타가 아니라 절대 명령이라 부호 반전이 안 됩니다. 대신
"역재생 기준 다음 프레임에서 손가락이 닫혀 있어야 하는가"로 다시 만듭니다.
그러면 원본의 *놓기* 순간이 역재생의 *잡기* 순간에 정확히 대응됩니다.


In [ ]:
!python scripts/build_dataset.py

In [ ]:
# 역재생이 제대로 됐는지 눈으로 확인
from vlapipe.episode import Episode
import glob
revs = sorted(glob.glob('data/episodes_reversed/*.hdf5'))
if revs:
    ep = Episode(revs[0])
    print(ep.path.name, '|', len(ep), '프레임 | task:', ep.task)
    idx = np.linspace(0, len(ep) - 1, 6).astype(int)
    fig, ax = plt.subplots(1, 6, figsize=(18, 3))
    for a, i in zip(ax, idx):
        a.imshow(ep.obs['agentview_image'][i]); a.axis('off'); a.set_title(f't={i}', fontsize=8)
    plt.suptitle('역재생된 정돈 데모 (왼쪽=시작, 오른쪽=끝)'); plt.tight_layout(); plt.show()
    imageio.mimsave('outputs/reversed.gif', list(ep.obs['agentview_image'][::3]), fps=12, loop=0)
    display(IPyImage(filename='outputs/reversed.gif'))
else:
    print('아직 녹화 데이터가 없습니다. 윈도우에서 scripts/teleop_record.py 를 먼저 실행하세요.')

---
## SECTION 7 | LeRobot 변환 & SmolVLA 파인튜닝

> 이 절은 **데이터를 모은 뒤**에 실행합니다. 에피소드 20~50개는 있어야 의미가 있습니다.


In [ ]:
# 7-1. LeRobot 설치 (시간이 좀 걸립니다)
!pip install -q 'lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git'

In [ ]:
# 7-2. LeRobot 데이터셋으로 변환
!python scripts/build_dataset.py --lerobot --repo-id dshs2026/vla_sorting

In [ ]:
# 7-3. SmolVLA 파인튜닝
#   base 모델 lerobot/smolvla_base 에서 출발합니다.
#   steps 는 처음엔 작게(2000) 잡고 학습곡선을 보며 늘리세요.
!python -m lerobot.scripts.train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=dshs2026/vla_sorting \
  --dataset.root=data/lerobot \
  --batch_size=8 --steps=2000 \
  --output_dir=outputs/smolvla_sorting \
  --job_name=vla_sorting --wandb.enable=false

In [ ]:
# 7-4. 학습된 정책으로 파이프라인 재실행
!python scripts/run_pipeline.py --policy smolvla \
  --checkpoint outputs/smolvla_sorting/checkpoints/last/pretrained_model \
  --layout messy --seed 11

---
## SECTION 8 | 미니과제

### 필수
1. **텔레옵 녹화 5개** — 정돈된 씬을 어지르는 장면. `build_dataset.py` 검증을 통과할 것
   (던지거나 떨어뜨린 구간이 있으면 자동 제외됩니다).
2. **프롬프트 배틀** — 팀원 3명이 서로 다른 프롬프트로 SECTION 3 채점 루프를 돌리고,
   5개 씬 평균 recall/precision 비교표를 만들 것. 가장 좋은 프롬프트를 고른 이유 1줄.
3. **실패 분석** — SECTION 4 의 실패 사유(`grasp_failed`, `dropped_during_carry`,
   `stalled_in_*`)를 물체 종류별로 집계하고, 왜 그 물체가 어려운지 2~3문장.

### 생각할 질문
- 지금 `agentview` 는 천장에서 내려다봅니다. 실제 로봇으로 옮길 때 이 시점을 못 쓴다면?
- scripted teacher 는 물체 좌표를 **직접 읽습니다**. SmolVLA 는 이미지만 봅니다.
  teacher 가 만든 데모로 학습한 정책이 teacher 만큼 잘할 수 있을까요? 무엇이 부족할까요?
- 역재생으로 만든 데모는 사람이 직접 정돈한 데모와 무엇이 다를까요?
  (힌트: 속도 분포, 접근 각도, 실패 후 재시도의 유무)


In [ ]:
# 제출용 저장
import json, shutil, datetime
TEAM = Path('/content/drive/MyDrive/VLA_Team_2026') if IN_COLAB else PKG.parent
dst = TEAM / 'results' / f'session12_{datetime.datetime.now():%m%d_%H%M}'
dst.mkdir(parents=True, exist_ok=True)
for p in Path('outputs').glob('**/*'):
    if p.is_file() and p.suffix in {'.gif', '.png', '.json'}:
        shutil.copy(p, dst / f'{p.parent.name}_{p.name}')
print('저장:', dst)
print(sorted(x.name for x in dst.iterdir()))